In [8]:
import pandas as pd
import string
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer
# from jobable.ml_logic.load_data import load_data
# from jobable.ml_logic.preprocess import add_bag_of_words_column
# from jobable.ml_logic.frequency import get_wordcounts
# from jobable.ml_logic.matching import compute_tfidf_similarity
# from jobable.ml_logic.recommendation import recommend_skills
from transformers import pipeline
import transformers
import torch
from transformers import AutoTokenizer
from jobable.ml_logic.data_keywords import DATA_KEYWORDS
import re
from nltk.corpus import stopwords
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Create DataFrames

In [15]:
df = pd.read_csv("/Users/jonny/code/JonnyBeAverage/Jobable/data/Jobs.csv")
df = df[['title', 'description']]


In [16]:
single_jd_description = df.iloc[0]['description']

In [17]:
resume_df = pd.read_csv("/Users/jonny/code/JonnyBeAverage/Jobable/data/Resume.csv")


In [18]:
single_resume = resume_df.iloc[0]
single_resume_description = single_resume['Resume_str']
single_resume_description

"         HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss pr

# Generate Cover Letter

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 1675.55it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


<pad> The position you are applying for is a data science position. The job description is as follows:


In [108]:
def truncate_to_token_limit(text, max_tokens=450):
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=max_tokens,
        return_tensors="pt"
    )
    return tokenizer.decode(tokens["input_ids"][0], skip_special_tokens=True)

In [5]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-large"
)


KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [ ]:
def summarize_text(text: str, max_tokens: int = 200):
    safe_text = truncate_to_token_limit(text)

    prompt = f"""
    Summarize the following text into concise professional bullet points.
    Focus on key skills, experience, and measurable achievements.

    {safe_text}
    """

    summarized_text = generator(prompt, max_new_tokens=max_tokens)
    return summarized_text[0]["generated_text"]

In [111]:
def create_cover_letter(cv_text: str, jd_text: str):

    summarized_cv = summarize_text(cv_text)
    summarized_jd = summarize_text(jd_text)

    raw_prompt = f"""
    You are a professional executive career coach.

    Write a tailored cover letter using the structure below.

    Structure:
    Paragraph 1:
    - Express interest in the specific role.
    - Mention years of leadership experience.

    Paragraph 2:
    - Connect 2-3 specific achievements to the job requirements.
    - Be concrete and avoid repetition.

    Paragraph 3:
    - Reinforce cultural fit and leadership strengths.

    Paragraph 4:
    - Confident closing and call to action.

    Rules:
    - Begin with: Dear Hiring Manager,
    - Do not repeat phrases.
    - Avoid generic language.
    - Do not restate the resume summary.
    - Keep under 300 words.

    Candidate Highlights:
    {summarized_cv}

    Role Requirements:
    {summarized_jd}

    Write the full letter now.
    """

    safe_prompt = truncate_to_token_limit(raw_prompt)

    result = generator(
        safe_prompt,
        max_new_tokens=350,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
    )
    return result[0]["generated_text"]

In [ ]:
print(create_cover_letter(single_resume_description, single_jd_description))

Dear Hiring Manager, I would like to invite you to review my resume and give me the opportunity to meet with you to discuss the role of Customer Service Manager at Neustar. My resume is attached for your review. I have 15+ years of experience in Hospitality and Customer Service Management. My objective is to assist you with a career that is focused on growing the business and creating a better world. I look forward to hearing from you. I am confident that I can be of service to you. I am confident that you will be successful in this position. Please contact me with your resume. Thank you.


# Cover Letter generator pt2

In [12]:
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")

input_text = "write a cover letter for a data science position"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

outputs = model.generate(input_ids)
print(tokenizer.decode(outputs[0]))

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 1447.44it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


<pad> The position you are applying for is a data science position. The job description is as follows:


In [20]:
def truncate_to_token_limit(text, max_tokens=450):
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=max_tokens,
        return_tensors="pt"
    )
    return tokenizer.decode(tokens["input_ids"][0], skip_special_tokens=True)

In [26]:
def summarize_text(text: str, max_tokens: int = 200):
    safe_text = truncate_to_token_limit(text)

    prompt = f"""
    Summarize the following text into concise professional bullet points.
    Focus on key skills, experience, and measurable achievements.

    {safe_text}
    """

    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    outputs = model.generate(input_ids, max_new_tokens=max_tokens)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def create_cover_letter(cv_text: str, jd_text: str):

    summarized_cv = summarize_text(cv_text)
    summarized_jd = summarize_text(jd_text)

    input_text = f"""
    You are a professional executive career coach.

    Write a tailored cover letter using the structure below.

    Structure:
    Paragraph 1:
    - Express interest in the specific role.
    - Mention years of leadership experience.

    Paragraph 2:
    - Connect 2-3 specific achievements to the job requirements.
    - Be concrete and avoid repetition.

    Paragraph 3:
    - Reinforce cultural fit and leadership strengths.

    Paragraph 4:
    - Confident closing and call to action.

    Rules:
    - Begin with: Dear Hiring Manager,
    - Do not repeat phrases.
    - Avoid generic language.
    - Do not restate the resume summary.
    - Keep under 300 words.

    Candidate Highlights:
    {summarized_cv}

    Role Requirements:
    {summarized_jd}

    Write the full letter now.
    """
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids
    outputs = model.generate(input_ids, max_new_tokens=350)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [28]:
create_cover_letter(single_resume_description, single_jd_description)

'Dear Hiring Manager, I am writing to inform you that I am currently seeking a Customer Service Manager to join the Neustar team. I have 15+ years of experience in customer service and hospitality management. I am a dedicated builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service. I have a strong track record of customer service excellence and a proven track record of delivering results. I am a skilled multi-tasker and a skilled trainer for cross server hospitality systems such as Hilton OnQ , Micros Opera PMS , Fidelio OPERA Reservation System ( ORS )'

# Process data_keywords

In [ ]:
def normalize_keywords(keywords):
    keywords = keywords.lower()
    keywords = re.sub(r"[^\w\s+#]", " ", keywords)
    # keep + and # for things like c++ and c#
    keywords = re.sub(r"\s+", " ", keywords)
    return keywords.strip()


In [69]:
NORMALIZED_KEYWORDS = {normalize_keywords(k) for k in DATA_KEYWORDS}


In [70]:
def generate_ngrams(words, n):
    return [" ".join(words[i:i+n]) for i in range(len(words)-n+1)]

In [73]:
def extract_keywords(text: str, keywords: set) -> set:
    text = normalize_keywords(text)
    words = text.split()

    matches = set()

    # Check unigrams, bigrams, trigrams
    for n in [1, 2, 3]:
        ngrams = generate_ngrams(words, n)
        for phrase in ngrams:
            if phrase in keywords:
                matches.add(phrase)

    return matches

In [74]:
extract_keywords(single_jd_description, DATA_KEYWORDS)

{'bayesian statistics',
 'clustering',
 'excel',
 'forecasting',
 'logistic regression',
 'metrics',
 'optimization',
 'probability',
 'python',
 'r',
 'regression',
 'ridge regression',
 'segmentation',
 'sql'}

In [77]:
job_df = load_data('/Users/jonny/code/JonnyBeAverage/Jobable/data/Jobs.csv')
test_job_instance = job_df.iloc[9, 2]
test_job_instance

'Lumicity'

In [79]:
resume_df = load_data('/Users/jonny/code/JonnyBeAverage/Jobable/data/Resume.csv')
test_resume_instance = resume_df.iloc[5,1]
test_resume_instance

'         HR GENERALIST       Summary     Dedicated and focused Administrative Assistant who excels at prioritizing, completing multiple tasks simultaneously and following through to achieve project goals. Seeking a role of increased responsibility and authority.       Highlights         Microsoft Office proficiency  Excel spreadsheets  Meticulous attention to detail  Results-oriented  Self-directed      Time management  Professional and mature  Self-starter  Legal administrative support            Experience     11/2008   to   08/2014     HR Generalist    Company Name   －   City  ,   State      Managed visa related employment processes for all non-immigrant faculty and staff.  Improved productivity and enhanced visa related services.  Improved operational structure by developing guidelines and tools for internal and external administration of non-immigrant employment procedures  Reduced internal employment authorization processing times by approximately 30 percent.  Prepared, reviewed

In [104]:
STOP_WORDS = set(stopwords.words("english"))
def preprocess_text(text):

    text = text.strip().lower()

    # Keep + and # (for c++, c#)
    text = re.sub(r"[^\w\s+#'-]", " ", text)
    words = text.split()

    matches = []

    # Find longest keyword length dynamically
    max_len = max(len(k.split()) for k in DATA_KEYWORDS)

    # Check phrases from 1 word up to max_len
    for n in range(1, max_len + 1):
        for i in range(len(words) - n + 1):
            phrase = " ".join(words[i:i+n])
            if phrase in DATA_KEYWORDS:
                matches.append(phrase)

    return set(matches)

In [105]:
text = "Hello World if the but I'm in not caring Python SQL c++ C++ C# machine learning engineer scikit-learn ggplot2 ci/cd master's in computer science google professional machine learning engineer"
result = preprocess_text(text)
result

{'c#',
 'c++',
 'ggplot2',
 'google professional machine learning engineer',
 'machine learning engineer',
 "master's in computer science",
 'python',
 'scikit-learn',
 'sql'}